# 05 · Experimento adicional y análisis de errores
**Entrega 2 · días 11–13.** Necesita GPU y E0–E3 ejecutados. La hipótesis del experimento adicional debe estar escrita antes.

In [ ]:
# PREPARACIÓN (igual en todos los notebooks): conecta Drive, entra a la carpeta del proyecto y deja listos los datos.
from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/yuca-comparacion'   # carpeta del proyecto dentro de su Google Drive

import os, sys
os.chdir(PROYECTO)
sys.path.insert(0, PROYECTO)

from src.datos import descargar_dataset
# El .zip se guarda una sola vez en Drive; las fotos se descomprimen en el disco de Colab (más rápido).
RAIZ = str(descargar_dataset(destino='/content/datos', carpeta_zip=PROYECTO + '/data'))
os.environ['YUCA_DATOS'] = RAIZ
print('Proyecto:', os.getcwd())
print('Datos   :', RAIZ)

In [ ]:
import torch
print('GPU disponible:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU: menú Entorno de ejecución > Cambiar tipo de entorno > T4 GPU')

## Experimento adicional · opción A: robustez al brillo
No se entrena nada. Se evalúan E1 y E3 sobre **copias** del mismo conjunto de prueba con el brillo multiplicado por cada factor.
Lo demás no cambia. Factor 1.0 debe reproducir exactamente el resultado original (comprobación).

In [ ]:
from src import adicional, analisis
from IPython.display import Image as Mostrar, display
import pandas as pd
tabla = adicional.ejecutar(('E1', 'E3'))
display(Mostrar('resultados/adicional_brillo/robustez_brillo.png'))
tabla

## Experimento adicional · opción B: augmentation moderado (solo si el equipo elige esta)
Repite E1 con las mismas condiciones, pero las fotos de **entrenamiento** reciben recortes, giros y volteos leves.
Validación y prueba no se tocan. El profesor exige una de las dos opciones; hacer ambas es decisión del equipo.

In [ ]:
EJECUTAR_AUGMENTATION = False   # <- decisión del equipo
if EJECUTAR_AUGMENTATION:
    from src.entrenar import ejecutar
    ejecutar('E1_aug')
    display(analisis.tabla_consolidada(('E1', 'E1_aug')).T)
    display(analisis.tabla_por_clase(('E1', 'E1_aug'), 'f1'))

## Confusiones más frecuentes de cada modelo

In [ ]:
for exp in ('E1', 'E2', 'E3'):
    print(exp)
    display(analisis.pares_confundidos(exp))

## Inspección de errores reales (mínimo 8)
Se muestran los errores con mayor confianza de cada modelo (criterio fijo, para no escoger a conveniencia).
Para cada uno, el equipo escribe su hipótesis en `docs/04_analisis_errores.md`: ¿confusión entre síntomas parecidos?, ¿fondo?, ¿iluminación?, ¿foto borrosa?, ¿lesión pequeña?, ¿etiqueta dudosa?

In [ ]:
EXPERIMENTO = 'E1'
errores = analisis.galeria_errores(EXPERIMENTO, n=8)
display(Mostrar(f'resultados/{EXPERIMENTO}/galeria_errores.png'))
errores[['ruta', 'real', 'predicha', 'confianza']]

In [ ]:
errores3 = analisis.galeria_errores('E3', n=8)
display(Mostrar('resultados/E3/galeria_errores.png'))
errores3[['ruta', 'real', 'predicha', 'confianza']]

In [ ]:
# ¿Se equivocan E1 y E3 en las mismas fotos?
a = pd.read_csv('resultados/E1/predicciones_test.csv').set_index('ruta')['acierto']
b = pd.read_csv('resultados/E3/predicciones_test.csv').set_index('ruta')['acierto']
pd.crosstab(a.rename('E1 acierta'), b.rename('E3 acierta'))

## Grad-CAM (opcional, recomendado para 4 imágenes)
El mapa de calor muestra qué zonas influyeron más en la respuesta. **No demuestra causalidad** ni que el modelo haya
reconocido la enfermedad: solo indica dónde hubo más activación para esa clase.

In [ ]:
for exp in ('E1', 'E3'):
    display(Mostrar(str(analisis.gradcam_galeria(exp, n=4))))

## Tabla final

In [ ]:
analisis.graficas_comparacion().T